In [132]:
import pandas as pd
import numpy as np
from statsmodels.tsa.api import VAR
from statsmodels.tsa.statespace.varmax import VARMAX

### Data cleaning 

In [10]:
gdp = pd.read_csv("GDP_Component.csv")

In [12]:
gdp.columns

Index(['REF_DATE', 'GEO', 'DGUID', 'Prices', 'Seasonal adjustment',
       'Estimates', 'UOM', 'UOM_ID', 'SCALAR_FACTOR', 'SCALAR_ID', 'VECTOR',
       'COORDINATE', 'VALUE', 'STATUS', 'SYMBOL', 'TERMINATED', 'DECIMALS'],
      dtype='object')

In [13]:
gdp['Estimates'].unique(), gdp['Estimates'].nunique()

(array(['Final consumption expenditure',
        'Household final consumption expenditure', 'Goods',
        'Durable goods', 'Semi-durable goods', 'Non-durable goods',
        'Services',
        "Non-profit institutions serving households' final consumption expenditure",
        'General governments final consumption expenditure',
        'Gross fixed capital formation',
        'Business gross fixed capital formation', 'Residential structures',
        'Non-residential structures, machinery and equipment',
        'Non-residential structures', 'Machinery and equipment',
        'Intellectual property products',
        "Non-profit institutions serving households' gross fixed capital formation",
        'General governments gross fixed capital formation',
        'Investment in inventories',
        'Of which: business investment in inventories', 'Non-farm', 'Farm',
        'Exports of goods and services', 'Exports of goods',
        'Exports of services', 'Less: imports of goods and

In [14]:
c = gdp[
    (gdp['GEO'] == 'Canada') &
    (gdp['Estimates'].isin([
        'Household final consumption expenditure',
        "Non-profit institutions serving households' final consumption expenditure"
    ]))
]

c = c.pivot_table(index='REF_DATE', columns='Estimates', values='VALUE').reset_index()
c['Consumption'] = c.sum(axis=1, numeric_only=True)

In [15]:
g = gdp[
    (gdp['GEO'] == 'Canada') &
    (gdp['Estimates'] == 'General governments final consumption expenditure')
]

g = g[['REF_DATE', 'VALUE']].rename(columns={'VALUE': 'Government'})

In [16]:
i = gdp[
    (gdp['GEO'] == 'Canada') &
    (gdp['Estimates'] == 'Gross fixed capital formation')
][['REF_DATE', 'VALUE']].rename(columns={'VALUE': 'Investment'})

x = gdp[
    (gdp['GEO'] == 'Canada') &
    (gdp['Estimates'] == 'Exports of goods and services')
][['REF_DATE', 'VALUE']].rename(columns={'VALUE': 'Exports'})

m = gdp[
    (gdp['GEO'] == 'Canada') &
    (gdp['Estimates'] == 'Less: imports of goods and services')
][['REF_DATE', 'VALUE']].rename(columns={'VALUE': 'Imports'})

In [18]:
df_gdp = c[['REF_DATE', 'Consumption']] \
    .merge(g, on='REF_DATE') \
    .merge(i, on='REF_DATE') \
    .merge(x, on='REF_DATE') \
    .merge(m, on='REF_DATE')

In [27]:
for col in ['Consumption', 'Government', 'Investment', 'Exports', 'Imports']:
    df_gdp[f'{col}_growth'] = df_gdp[col].pct_change() * 100

In [28]:
df_gdp = df_gdp.rename(columns={'REF_DATE': 'quarter'})
df_gdp.head()

,quarter,Consumption,Government,Investment,Exports,Imports,Consumption_growth,Government_growth,Investment_growth,Exports_growth,Imports_growth
0,1990-01,599337.0,302683,251583,247653,230451,NaN,NaN,NaN,NaN,NaN
1,1990-04,592638.0,299097,245503,256667,231011,-1.117735,-1.184738,-2.416697,3.639770,0.243002
2,1990-07,592941.0,305704,238767,255179,226643,0.051127,2.208982,-2.743755,-0.579740,-1.890819
3,1990-10,591024.0,309111,230205,248406,223608,-0.323304,1.114477,-3.585923,-2.654215,-1.339110
4,1991-01,579203.0,307509,232129,243338,224153,-2.000088,-0.518260,0.835777,-2.040208,0.243730


In [41]:

cpi = cpi[(cpi['GEO'] == 'Canada') & (cpi['Products and product groups'] == 'All-items')]

cpi['quarter'] = pd.to_datetime(cpi['REF_DATE']).dt.to_period('Q')
cpi_quarterly = cpi.groupby('quarter')['VALUE'].mean().reset_index()
cpi_quarterly = cpi_quarterly.rename(columns={'VALUE': 'CPI'})
cpi_quarterly['CPI_growth'] = cpi_quarterly['CPI'].pct_change() * 100

cpi_quarterly = cpi_quarterly[['quarter', 'CPI', 'CPI_growth']]
cpi_quarterly.head()
cpi_quarterly.to_csv("cpi.csv", index=False)


In [42]:
cpi_quarterly.head()

,quarter,CPI,CPI_growth
0,1992Q1,83.300000,NaN
1,1992Q2,83.900000,0.720288
2,1992Q3,84.200000,0.357569
3,1992Q4,84.533333,0.395883
4,1993Q1,85.100000,0.670347


In [30]:
pop = pd.read_csv("Population.csv")
pop.head()

,REF_DATE,GEO,DGUID,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,1990-01,Canada,2021A000011124,Persons,249,units,0,v1,1,27463550,NaN,NaN,NaN,0
1,1990-04,Canada,2021A000011124,Persons,249,units,0,v1,1,27567161,NaN,NaN,NaN,0
2,1990-07,Canada,2021A000011124,Persons,249,units,0,v1,1,27691138,NaN,NaN,NaN,0
3,1990-10,Canada,2021A000011124,Persons,249,units,0,v1,1,27807591,NaN,NaN,NaN,0
4,1991-01,Canada,2021A000011124,Persons,249,units,0,v1,1,27854861,NaN,NaN,NaN,0


In [31]:
pop = pop[pop['GEO'] == 'Canada']
pop = pop[['REF_DATE', 'VALUE']].rename(columns={'REF_DATE': 'quarter', 'VALUE': 'Population'})
pop['Population_growth'] = pop['Population'].pct_change() * 100

In [33]:
pop = pop[['quarter', 'Population', 'Population_growth']]
pop.head()

,quarter,Population,Population_growth
0,1990-01,27463550,NaN
1,1990-04,27567161,0.377267
2,1990-07,27691138,0.449727
3,1990-10,27807591,0.420542
4,1991-01,27854861,0.169990


In [34]:
pop.to_csv("population.csv", index=False)

In [ ]:

unemp = pd.read_csv("unemployment.csv")

unemp.columns = ['date', 'Unemployment_rate']

unemp['quarter'] = pd.to_datetime(unemp['date']).dt.to_period('Q')

unemp['Unemp_growth_pct'] = unemp['Unemployment_rate'].pct_change() * 100

unemp = unemp[['quarter', 'Unemployment_rate', 'Unemp_growth_pct']]
unemp.to_csv("unemployment.csv", index=False)

In [46]:
unemp.head()

,quarter,Unemployment_rate,Unemp_growth_pct
0,1955Q1,4.800000,NaN
1,1955Q2,4.433333,-7.638896
2,1955Q3,4.066667,-8.270662
3,1955Q4,3.833333,-5.737721
4,1956Q1,3.700000,-3.478252


In [49]:
com_pi = pd.read_csv("commdity_price_index.csv")
com_pi.head()

,date,M.BCPI,M.BCNE,M.ENER,M.MTLS,M.FOPR,M.AGRI,M.FISH,quarter
0,1972-03-31,100.503333,100.626667,99.943333,100.706667,100.120000,101.243333,95.953333,1972Q1
1,1972-06-30,101.713333,102.093333,100.000000,101.110000,102.070000,103.086667,93.366667,1972Q2
2,1972-09-30,104.770000,105.633333,100.913333,101.543333,105.146667,109.183333,101.463333,1972Q3
3,1972-12-31,108.200000,109.733333,101.460000,102.443333,106.986667,117.880000,116.536667,1972Q4
4,1973-03-31,118.013333,121.170000,104.523333,113.046667,115.163333,134.493333,116.193333,1973Q1


In [ ]:
cols_to_diff = [col for col in com_pi.columns if col not in ['date', 'quarter']]

for col in cols_to_diff:
    com_pi[f'{col}_growth'] = com_pi[col].pct_change() * 100

com_pi = com_pi.drop(columns=['date'])

In [52]:
com_pi.head()
com_pi.to_csv("commodity_price_index.csv", index=False)

In [53]:
us_gdp = pd.read_csv("us_gdp_real.csv")

In [54]:
us_gdp.head()

,observation_date,GDPC1
0,1947-01-01,2182.681
1,1947-04-01,2176.892
2,1947-07-01,2172.432
3,1947-10-01,2206.452
4,1948-01-01,2239.682


In [55]:
us_gdp['quarter'] = pd.to_datetime(us_gdp['observation_date']).dt.to_period('Q')


us_gdp['US_GDP_growth'] = us_gdp['GDPC1'].pct_change() * 100

us_gdp = us_gdp[['quarter', 'GDPC1', 'US_GDP_growth']]
us_gdp.head()
us_gdp.to_csv("us_gdp.csv", index=False)

In [56]:
us_trade = pd.read_excel("USTradePolicyIndex.xlsx")

In [58]:

us_trade = pd.read_excel("USTradePolicyIndex.xlsx", sheet_name="TPU_QUARTERLY")


us_trade = us_trade.rename(columns={"DATEQ": "quarter"})


us_trade['TPUQ_growth'] = us_trade['TPUQ'].pct_change() * 100
us_trade['TARIFFVOL_growth'] = us_trade['TARIFFVOL'].pct_change() * 100


us_trade = us_trade[['quarter', 'TPUQ', 'TPUQ_growth', 'TARIFFVOL', 'TARIFFVOL_growth']]

us_trade.head()

C:\Users\logc0\AppData\Local\Temp\ipykernel_34472\1107649623.py:8: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  us_trade['TARIFFVOL_growth'] = us_trade['TARIFFVOL'].pct_change() * 100


,quarter,TPUQ,TPUQ_growth,TARIFFVOL,TARIFFVOL_growth
0,1960Q1,14.949300,NaN,0.399707,NaN
1,1960Q2,16.928593,13.240038,0.546273,36.668435
2,1960Q3,15.502807,-8.422354,0.360788,-33.954666
3,1960Q4,16.523557,6.584291,0.336059,-6.854084
4,1961Q1,18.406929,11.398105,0.220606,-34.355028


### Merge to level and growth rate data

In [60]:
df_gdp['quarter'] = df_gdp['quarter'].astype(str)
cpi_quarterly['quarter'] = cpi_quarterly['quarter'].astype(str)
pop['quarter'] = pop['quarter'].astype(str)
unemp['quarter'] = unemp['quarter'].astype(str)
com_pi['quarter'] = com_pi['quarter'].astype(str)
us_gdp['quarter'] = us_gdp['quarter'].astype(str)
us_trade['quarter'] = us_trade['quarter'].astype(str)

In [77]:
level_data = df_gdp[['quarter', 'Consumption', 'Government', 'Investment', 'Exports', 'Imports']] \
    .merge(cpi_quarterly[['quarter', 'CPI']], on='quarter', how='inner') \
    .merge(pop[['quarter', 'Population']], on='quarter', how='inner') \
    .merge(unemp[['quarter', 'Unemployment_rate']], on='quarter', how='inner') \
    .merge(com_pi[['quarter', 'M.BCPI', 'M.BCNE', 'M.ENER', 'M.MTLS', 'M.FOPR', 'M.AGRI', 'M.FISH']], on='quarter', how='inner') \
    .merge(us_gdp[['quarter', 'GDPC1']], on='quarter', how='inner') \
    .merge(us_trade[['quarter', 'TPUQ', 'TARIFFVOL']], on='quarter', how='inner')

level_data.head()

,quarter,Consumption,Government,Investment,Exports,Imports,CPI,Population,Unemployment_rate,M.BCPI,M.BCNE,M.ENER,M.MTLS,M.FOPR,M.AGRI,M.FISH,GDPC1,TPUQ,TARIFFVOL
0,1992Q1,592278.0,313952,231808,265729,243333,83.300000,28181477,10.60000,246.386667,209.413333,429.476667,260.466667,240.893333,160.763333,599.166667,10236.435,42.225518,0.121545
1,1992Q2,594721.0,312184,222221,273246,245926,83.900000,28269699,11.00000,255.593333,214.206667,459.136667,263.570000,243.936667,168.550000,627.276667,10347.429,37.558292,0.094423
2,1992Q3,597899.0,312118,226708,277420,250009,84.200000,28371264,11.53333,257.000000,212.633333,474.433333,268.433333,244.690000,162.636667,534.963333,10449.673,50.093674,0.082377
3,1992Q4,600089.0,317134,217324,285010,244810,84.533333,28474177,11.73333,254.593333,209.013333,478.203333,246.290000,249.176667,162.053333,574.763333,10558.648,53.682379,0.057045
4,1993Q1,603297.0,314100,214916,295333,255921,85.100000,28533602,11.13333,266.456667,228.426667,455.486667,243.973333,301.050000,167.423333,672.156667,10576.275,50.748552,0.092414


In [68]:
for df in [df_gdp, cpi_quarterly, pop, unemp, com_pi, us_gdp, us_trade]:
    df['quarter'] = df['quarter'].astype(str).str.strip().str.upper()

In [70]:
for name, df in [
    ('df_gdp', df_gdp),
    ('cpi_quarterly', cpi_quarterly),
    ('pop', pop),
    ('unemp', unemp),
    ('com_pi', com_pi),
    ('us_gdp', us_gdp),
    ('us_trade', us_trade)
]:
    print(f"{name}: {list(df.columns)}")

df_gdp: ['quarter', 'Consumption', 'Government', 'Investment', 'Exports', 'Imports', 'Consumption_growth', 'Government_growth', 'Investment_growth', 'Exports_growth', 'Imports_growth']
cpi_quarterly: ['quarter', 'CPI', 'CPI_growth']
pop: ['quarter', 'Population', 'Population_growth']
unemp: ['quarter', 'Unemployment_rate', 'Unemp_growth_pct']
com_pi: ['M.BCPI', 'M.BCNE', 'M.ENER', 'M.MTLS', 'M.FOPR', 'M.AGRI', 'M.FISH', 'quarter', 'M.BCPI_growth', 'M.BCNE_growth', 'M.ENER_growth', 'M.MTLS_growth', 'M.FOPR_growth', 'M.AGRI_growth', 'M.FISH_growth']
us_gdp: ['quarter', 'GDPC1', 'US_GDP_growth']
us_trade: ['quarter', 'TPUQ', 'TPUQ_growth', 'TARIFFVOL', 'TARIFFVOL_growth']


In [75]:
print("df_gdp:", len(df_gdp['quarter'].unique()))
print("cpi_quarterly:", len(cpi_quarterly['quarter'].unique()))
print("pop:", len(pop['quarter'].unique()))
print("unemp:", len(unemp['quarter'].unique()))
print("com_pi:", len(com_pi['quarter'].unique()))
print("us_gdp:", len(us_gdp['quarter'].unique()))
print("us_trade:", len(us_trade['quarter'].unique()))

df_gdp: 142
cpi_quarterly: 135
pop: 141
unemp: 283
com_pi: 215
us_gdp: 314
us_trade: 263


In [76]:
quarters = set(df_gdp['quarter'])
quarters &= set(cpi_quarterly['quarter'])
quarters &= set(pop['quarter'])
quarters &= set(unemp['quarter'])
quarters &= set(com_pi['quarter'])
quarters &= set(us_gdp['quarter'])
quarters &= set(us_trade['quarter'])

print("Common quarters:", sorted(quarters))
print("Count:", len(quarters))

Common quarters: ['1992Q1', '1992Q2', '1992Q3', '1992Q4', '1993Q1', '1993Q2', '1993Q3', '1993Q4', '1994Q1', '1994Q2', '1994Q3', '1994Q4', '1995Q1', '1995Q2', '1995Q3', '1995Q4', '1996Q1', '1996Q2', '1996Q3', '1996Q4', '1997Q1', '1997Q2', '1997Q3', '1997Q4', '1998Q1', '1998Q2', '1998Q3', '1998Q4', '1999Q1', '1999Q2', '1999Q3', '1999Q4', '2000Q1', '2000Q2', '2000Q3', '2000Q4', '2001Q1', '2001Q2', '2001Q3', '2001Q4', '2002Q1', '2002Q2', '2002Q3', '2002Q4', '2003Q1', '2003Q2', '2003Q3', '2003Q4', '2004Q1', '2004Q2', '2004Q3', '2004Q4', '2005Q1', '2005Q2', '2005Q3', '2005Q4', '2006Q1', '2006Q2', '2006Q3', '2006Q4', '2007Q1', '2007Q2', '2007Q3', '2007Q4', '2008Q1', '2008Q2', '2008Q3', '2008Q4', '2009Q1', '2009Q2', '2009Q3', '2009Q4', '2010Q1', '2010Q2', '2010Q3', '2010Q4', '2011Q1', '2011Q2', '2011Q3', '2011Q4', '2012Q1', '2012Q2', '2012Q3', '2012Q4', '2013Q1', '2013Q2', '2013Q3', '2013Q4', '2014Q1', '2014Q2', '2014Q3', '2014Q4', '2015Q1', '2015Q2', '2015Q3', '2015Q4', '2016Q1', '2016Q2', '2

In [73]:
for name, df in [
    ('df_gdp', df_gdp),
    ('cpi_quarterly', cpi_quarterly),
    ('pop', pop),
    ('unemp', unemp),
    ('com_pi', com_pi),
    ('us_gdp', us_gdp),
    ('us_trade', us_trade)
]:
    print(f"{name}: type={df['quarter'].dtype}, example={df['quarter'].iloc[0]}")

df_gdp: type=object, example=1990-01
cpi_quarterly: type=object, example=1992Q1
pop: type=object, example=1990-01
unemp: type=object, example=1955Q1
com_pi: type=object, example=1972Q1
us_gdp: type=object, example=1947Q1
us_trade: type=object, example=1960Q1


In [74]:
def fix_quarter_format(df):
    df['quarter'] = pd.to_datetime(df['quarter'], errors='coerce') \
                        .dt.to_period('Q') \
                        .astype(str)
    return df

df_gdp = fix_quarter_format(df_gdp)
pop = fix_quarter_format(pop)

In [78]:
growth_data = df_gdp[['quarter', 'Consumption_growth', 'Government_growth', 'Investment_growth',
                      'Exports_growth', 'Imports_growth']] \
    .merge(cpi_quarterly[['quarter', 'CPI_growth']], on='quarter', how='inner') \
    .merge(pop[['quarter', 'Population_growth']], on='quarter', how='inner') \
    .merge(unemp[['quarter', 'Unemp_growth_pct']], on='quarter', how='inner') \
    .merge(com_pi[['quarter', 'M.BCPI_growth', 'M.BCNE_growth', 'M.ENER_growth',
                   'M.MTLS_growth', 'M.FOPR_growth', 'M.AGRI_growth', 'M.FISH_growth']], on='quarter', how='inner') \
    .merge(us_gdp[['quarter', 'US_GDP_growth']], on='quarter', how='inner') \
    .merge(us_trade[['quarter', 'TPUQ_growth', 'TARIFFVOL_growth']], on='quarter', how='inner')

growth_data.head()

,quarter,Consumption_growth,Government_growth,Investment_growth,Exports_growth,Imports_growth,CPI_growth,Population_growth,Unemp_growth_pct,M.BCPI_growth,M.BCNE_growth,M.ENER_growth,M.MTLS_growth,M.FOPR_growth,M.AGRI_growth,M.FISH_growth,US_GDP_growth,TPUQ_growth,TARIFFVOL_growth
0,1992Q1,0.030738,0.161112,-0.693579,0.879611,-0.123136,NaN,0.192517,2.580678,0.023004,5.234594,-10.427413,2.151803,5.126266,5.890748,50.990785,1.197252,8.989815,-10.815336
1,1992Q2,0.412475,-0.563143,-4.135750,2.828822,1.065618,0.720288,0.313050,3.773585,3.736674,2.288934,6.906079,1.191451,1.263353,4.843559,4.691516,1.084303,-11.053093,-22.314655
2,1992Q3,0.534368,-0.021141,2.019161,1.527561,1.660256,0.357569,0.359272,4.848455,0.550353,-0.734493,3.331615,1.845177,0.308823,-3.508356,-14.716526,0.988110,33.375802,-12.756711
3,1992Q4,0.366283,1.607085,-4.139245,2.735924,-2.079525,0.395883,0.362737,1.734105,-0.936446,-1.702461,0.794632,-8.249100,1.833613,-0.358673,7.439762,1.042856,7.163989,-30.751405
4,1993Q1,0.534587,-0.956693,-1.108023,3.621978,4.538622,0.670347,0.208698,-5.113638,4.659719,9.288084,-4.750420,-0.940626,20.817894,3.313724,16.944945,0.166944,-5.465158,62.002023


In [79]:
growth_data = growth_data.iloc[1:].reset_index(drop=True)
growth_data.head()

,quarter,Consumption_growth,Government_growth,Investment_growth,Exports_growth,Imports_growth,CPI_growth,Population_growth,Unemp_growth_pct,M.BCPI_growth,M.BCNE_growth,M.ENER_growth,M.MTLS_growth,M.FOPR_growth,M.AGRI_growth,M.FISH_growth,US_GDP_growth,TPUQ_growth,TARIFFVOL_growth
0,1992Q2,0.412475,-0.563143,-4.135750,2.828822,1.065618,0.720288,0.313050,3.773585,3.736674,2.288934,6.906079,1.191451,1.263353,4.843559,4.691516,1.084303,-11.053093,-22.314655
1,1992Q3,0.534368,-0.021141,2.019161,1.527561,1.660256,0.357569,0.359272,4.848455,0.550353,-0.734493,3.331615,1.845177,0.308823,-3.508356,-14.716526,0.988110,33.375802,-12.756711
2,1992Q4,0.366283,1.607085,-4.139245,2.735924,-2.079525,0.395883,0.362737,1.734105,-0.936446,-1.702461,0.794632,-8.249100,1.833613,-0.358673,7.439762,1.042856,7.163989,-30.751405
3,1993Q1,0.534587,-0.956693,-1.108023,3.621978,4.538622,0.670347,0.208698,-5.113638,4.659719,9.288084,-4.750420,-0.940626,20.817894,3.313724,16.944945,0.166944,-5.465158,62.002023
4,1993Q2,0.364497,0.426934,2.376743,2.864563,3.143548,0.313357,0.235729,4.491019,-3.107447,-4.967313,1.094068,-1.080719,-11.393456,1.377745,-7.313771,0.582171,-11.349354,70.120623


In [80]:
growth_data.to_csv("growth_data.csv", index=False)
level_data.to_csv("level_data.csv", index=False)

In [171]:
growth = pd.read_csv("growth_data.csv")
level = pd.read_csv("level_data.csv")

In [143]:
growth.columns

Index(['quarter', 'Consumption_growth', 'Government_growth',
       'Investment_growth', 'Exports_growth', 'Imports_growth', 'CPI_growth',
       'Population_growth', 'Unemp_growth_pct', 'M.BCPI_growth',
       'M.BCNE_growth', 'M.ENER_growth', 'M.MTLS_growth', 'M.FOPR_growth',
       'M.AGRI_growth', 'M.FISH_growth', 'US_GDP_growth', 'TPUQ_growth',
       'TARIFFVOL_growth'],
      dtype='object')

In [144]:
level.columns

Index(['quarter', 'Consumption', 'Government', 'Investment', 'Exports',
       'Imports', 'CPI', 'Population', 'Unemployment_rate', 'M.BCPI', 'M.BCNE',
       'M.ENER', 'M.MTLS', 'M.FOPR', 'M.AGRI', 'M.FISH', 'GDPC1', 'TPUQ',
       'TARIFFVOL'],
      dtype='object')

In [172]:
# overwrite data in growth using level, matched by 'quarter'
cols_map = {
    'Unemp_growth_pct': 'Unemployment_rate',
    'TPUQ_growth': 'TPUQ',
    'TARIFFVOL_growth': 'TARIFFVOL'
}

g = growth.set_index('quarter').copy()
lv = level.set_index('quarter')

for gcol, lcol in cols_map.items():
    g[gcol] = lv[lcol].reindex(g.index).values

growth_updated = g.reset_index()
growth_updated = growth_updated.rename(columns={
    'Unemp_growth_pct': 'Unemployment_rate',
    'TPUQ_growth': 'TPUQ',
    'TARIFFVOL_growth': 'TARIFFVOL'
})
growth_updated.head()

,quarter,Consumption_growth,Government_growth,Investment_growth,Exports_growth,Imports_growth,CPI_growth,Population_growth,Unemployment_rate,M.BCPI_growth,M.BCNE_growth,M.ENER_growth,M.MTLS_growth,M.FOPR_growth,M.AGRI_growth,M.FISH_growth,US_GDP_growth,TPUQ,TARIFFVOL
0,1992Q2,0.412475,-0.563143,-4.135750,2.828822,1.065618,0.720288,0.313050,11.00000,3.736674,2.288934,6.906079,1.191451,1.263353,4.843559,4.691516,1.084303,37.558292,0.094423
1,1992Q3,0.534368,-0.021141,2.019161,1.527561,1.660256,0.357569,0.359272,11.53333,0.550353,-0.734493,3.331615,1.845177,0.308823,-3.508356,-14.716526,0.988110,50.093674,0.082377
2,1992Q4,0.366283,1.607085,-4.139245,2.735924,-2.079525,0.395883,0.362737,11.73333,-0.936446,-1.702461,0.794632,-8.249100,1.833613,-0.358673,7.439762,1.042856,53.682379,0.057045
3,1993Q1,0.534587,-0.956693,-1.108023,3.621978,4.538622,0.670347,0.208698,11.13333,4.659719,9.288084,-4.750420,-0.940626,20.817894,3.313724,16.944945,0.166944,50.748552,0.092414
4,1993Q2,0.364497,0.426934,2.376743,2.864563,3.143548,0.313357,0.235729,11.63333,-3.107447,-4.967313,1.094068,-1.080719,-11.393456,1.377745,-7.313771,0.582171,44.988920,0.157216


In [174]:
# ensure both keys are the same string format like '1992Q2'
for _df in (growth_updated, tbill_q):
    _df["quarter"] = pd.PeriodIndex(_df["quarter"], freq="Q").astype(str)

# inner merge
growth_with_rate = growth_updated.merge(tbill_q, on="quarter", how="inner")

print(growth_with_rate.shape)
growth_with_rate.head()

(101, 20)


,quarter,Consumption_growth,Government_growth,Investment_growth,Exports_growth,Imports_growth,CPI_growth,Population_growth,Unemployment_rate,M.BCPI_growth,M.BCNE_growth,M.ENER_growth,M.MTLS_growth,M.FOPR_growth,M.AGRI_growth,M.FISH_growth,US_GDP_growth,TPUQ,TARIFFVOL,tbill_3m
0,2000Q1,1.065801,1.134015,0.897780,3.329395,2.131252,0.639886,0.110737,6.866667,7.957180,4.561768,11.019321,4.580602,3.830421,6.177863,2.366356,0.362793,32.377237,0.156286,5.27
1,2000Q2,0.865105,1.102124,0.682871,1.483610,0.575793,0.459202,0.223279,6.666667,5.380915,-2.772814,16.407504,-4.378775,-5.572960,3.933478,2.610906,1.821288,37.325161,0.148466,5.53
2,2000Q3,1.281944,0.467634,1.201781,0.827234,0.979522,0.914205,0.299732,6.900000,4.086174,-4.956893,15.686098,1.065243,-8.463894,-4.017511,-5.350437,0.101933,31.529603,0.138617,5.56
3,2000Q4,0.234857,0.610294,0.450685,0.178956,-0.874908,1.080139,0.320146,6.900000,5.740800,-3.221648,17.439389,-3.608452,-3.716544,-1.716507,-5.043168,0.597039,27.400892,0.107933,5.49
4,2001Q1,0.774141,1.280856,2.721848,-1.676192,-2.666061,0.241296,0.131471,7.000000,-5.848566,-1.048308,-10.278571,-0.832326,-2.013093,-0.196594,13.515337,-0.327799,23.904262,0.124375,4.58


In [175]:
growth_updated = growth_with_rate
growth_updated.to_csv("data.csv")

In [176]:
growth_updated.head()

,quarter,Consumption_growth,Government_growth,Investment_growth,Exports_growth,Imports_growth,CPI_growth,Population_growth,Unemployment_rate,M.BCPI_growth,M.BCNE_growth,M.ENER_growth,M.MTLS_growth,M.FOPR_growth,M.AGRI_growth,M.FISH_growth,US_GDP_growth,TPUQ,TARIFFVOL,tbill_3m
0,2000Q1,1.065801,1.134015,0.897780,3.329395,2.131252,0.639886,0.110737,6.866667,7.957180,4.561768,11.019321,4.580602,3.830421,6.177863,2.366356,0.362793,32.377237,0.156286,5.27
1,2000Q2,0.865105,1.102124,0.682871,1.483610,0.575793,0.459202,0.223279,6.666667,5.380915,-2.772814,16.407504,-4.378775,-5.572960,3.933478,2.610906,1.821288,37.325161,0.148466,5.53
2,2000Q3,1.281944,0.467634,1.201781,0.827234,0.979522,0.914205,0.299732,6.900000,4.086174,-4.956893,15.686098,1.065243,-8.463894,-4.017511,-5.350437,0.101933,31.529603,0.138617,5.56
3,2000Q4,0.234857,0.610294,0.450685,0.178956,-0.874908,1.080139,0.320146,6.900000,5.740800,-3.221648,17.439389,-3.608452,-3.716544,-1.716507,-5.043168,0.597039,27.400892,0.107933,5.49
4,2001Q1,0.774141,1.280856,2.721848,-1.676192,-2.666061,0.241296,0.131471,7.000000,-5.848566,-1.048308,-10.278571,-0.832326,-2.013093,-0.196594,13.515337,-0.327799,23.904262,0.124375,4.58


In [163]:
df.rename(columns={df.columns[0]: "date"}, inplace=True)
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# 3) Keep the chosen series and compute end-of-quarter (last business day in quarter)
rate = df[["date", series_id]].dropna(subset=["date"]).copy()

q = (rate.set_index("date")
          .groupby(pd.Grouper(freq="Q"))[series_id]
          .last()                 # end-of-quarter value
          .dropna()
          .reset_index())

# 4) Format the quarter label and keep exactly two columns
q["quarter"] = q["date"].dt.to_period("Q").astype(str)
tbill_q = q.drop(columns=["date"]).rename(columns={series_id: out_col})[["quarter", out_col]]

print(tbill_q.head())
print(tbill_q.tail())
print(tbill_q.shape)


  quarter  tbill_3m
0  2000Q1      5.27
1  2000Q2      5.53
2  2000Q3      5.56
3  2000Q4      5.49
4  2001Q1      4.58
    quarter  tbill_3m
99   2024Q4      3.15
100  2025Q1      2.64
101  2025Q2      2.67
102  2025Q3      2.42
103  2025Q4      2.23
(104, 2)


C:\Users\logc0\AppData\Local\Temp\ipykernel_34472\3451455066.py:8: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  .groupby(pd.Grouper(freq="Q"))[series_id]


### VAR forecast

In [81]:

exog_vars = ['TPUQ_growth', 'TARIFFVOL_growth', 'US_GDP_growth']


exog = growth_data[exog_vars]
endog = growth_data.drop(columns=exog_vars + ['quarter']) 


print("Endogenous variables:", endog.columns.tolist())
print("Exogenous variables:", exog.columns.tolist())

Endogenous variables: ['Consumption_growth', 'Government_growth', 'Investment_growth', 'Exports_growth', 'Imports_growth', 'CPI_growth', 'Population_growth', 'Unemp_growth_pct', 'M.BCPI_growth', 'M.BCNE_growth', 'M.ENER_growth', 'M.MTLS_growth', 'M.FOPR_growth', 'M.AGRI_growth', 'M.FISH_growth']
Exogenous variables: ['TPUQ_growth', 'TARIFFVOL_growth', 'US_GDP_growth']


In [146]:
from statsmodels.tsa.api import VAR


model = VAR(endog)
lag_order_results = model.select_order(maxlags=8)
print(lag_order_results.summary())

 VAR Order Selection (* highlights the minimums) 
      AIC         BIC         FPE         HQIC   
-------------------------------------------------
0       5.727      5.933*       307.2       5.811
1       4.522       6.580       92.36      5.358*
2       4.206       8.115       68.59       5.794
3       4.064       9.826       62.62       6.404
4       3.780       11.39       52.08       6.873
5       3.465       12.93      45.06*       7.310
6       3.273       14.59       48.63       7.870
7       3.112       16.28       62.05       8.461
8      2.628*       17.65       69.37       8.729
-------------------------------------------------


In [84]:
print("Number of observations:", len(endog))

Number of observations: 132


In [91]:
p = 1
aligned_exog = exog.iloc[p:].reset_index(drop=True)
aligned_endog = endog.iloc[p:].reset_index(drop=True)

In [99]:
import pandas as pd
from statsmodels.tsa.stattools import adfuller

all_series = pd.concat([endog, exog], axis=1)


stationarity_results = []

for column in all_series.columns:
    series = all_series[column].dropna()
    adf_result = adfuller(series, autolag='AIC')

    result = {
        'Variable': column,
        'ADF Statistic': adf_result[0],
        'p-value': adf_result[1],
        'Lags Used': adf_result[2],
        'Observations': adf_result[3],
        'Stationary': 'Yes' if adf_result[1] < 0.05 else 'No'
    }
    stationarity_results.append(result)


stationarity_df = pd.DataFrame(stationarity_results)

stationarity_df.to_csv('stationarity_test_results.csv', index=False)

print(stationarity_df.head())

             Variable  ADF Statistic       p-value  Lags Used  Observations  \
0  Consumption_growth     -11.223970  1.981583e-20          1           129   
1   Government_growth     -12.666922  1.268897e-23          0           130   
2   Investment_growth     -10.603601  6.082105e-19          0           130   
3      Exports_growth      -3.979756  1.520305e-03          4           126   
4      Imports_growth     -13.176754  1.218910e-24          0           130   

  Stationary  
0        Yes  
1        Yes  
2        Yes  
3        Yes  
4        Yes  


In [100]:
results_df = pd.read_csv("stationarity_test_results.csv")
non_stationary = results_df[results_df['p-value'] > 0.05]['Variable'].tolist()
stationary = results_df[results_df['p-value'] <= 0.05]['Variable'].tolist()
print("Non-stationary variables (p > 0.05):")
for var in non_stationary:
    print("-", var)
print("\nStationary variables (p ≤ 0.05):")
for var in stationary:
    print("-", var)

Non-stationary variables (p > 0.05):

Stationary variables (p ≤ 0.05):
- Consumption_growth
- Government_growth
- Investment_growth
- Exports_growth
- Imports_growth
- CPI_growth
- Unemp_growth_pct
- M.BCPI_growth
- M.BCNE_growth
- M.ENER_growth
- M.MTLS_growth
- M.FOPR_growth
- M.AGRI_growth
- M.FISH_growth
- Population_growth_diff
- TPUQ_growth_diff
- TARIFFVOL_growth
- US_GDP_growth


In [97]:

growth_data['Population_growth_diff'] = growth_data['Population_growth'].diff()
growth_data['TPUQ_growth_diff'] = growth_data['TPUQ_growth'].diff()

growth_data = growth_data.drop(columns=['Population_growth', 'TPUQ_growth'])


growth_data = growth_data.dropna().reset_index(drop=True)

growth_data.head()

,quarter,Consumption_growth,Government_growth,Investment_growth,Exports_growth,Imports_growth,CPI_growth,Unemp_growth_pct,M.BCPI_growth,M.BCNE_growth,M.ENER_growth,M.MTLS_growth,M.FOPR_growth,M.AGRI_growth,M.FISH_growth,US_GDP_growth,TARIFFVOL_growth,Population_growth_diff,TPUQ_growth_diff
0,1992Q3,0.534368,-0.021141,2.019161,1.527561,1.660256,0.357569,4.848455,0.550353,-0.734493,3.331615,1.845177,0.308823,-3.508356,-14.716526,0.988110,-12.756711,0.046222,44.428895
1,1992Q4,0.366283,1.607085,-4.139245,2.735924,-2.079525,0.395883,1.734105,-0.936446,-1.702461,0.794632,-8.249100,1.833613,-0.358673,7.439762,1.042856,-30.751405,0.003465,-26.211813
2,1993Q1,0.534587,-0.956693,-1.108023,3.621978,4.538622,0.670347,-5.113638,4.659719,9.288084,-4.750420,-0.940626,20.817894,3.313724,16.944945,0.166944,62.002023,-0.154039,-12.629147
3,1993Q2,0.364497,0.426934,2.376743,2.864563,3.143548,0.313357,4.491019,-3.107447,-4.967313,1.094068,-1.080719,-11.393456,1.377745,-7.313771,0.582171,70.120623,0.027031,-5.884196
4,1993Q3,0.660781,-1.155842,2.167945,0.767957,0.577726,0.351425,-1.719198,-3.782939,-2.244948,-7.092029,-0.272096,-5.030928,1.304032,-16.639468,0.477155,-20.942747,0.057619,22.273635


In [177]:
growth_data = pd.read_csv("data.csv")

In [179]:
growth_data.columns

Index(['Unnamed: 0', 'quarter', 'Consumption_growth', 'Government_growth',
       'Investment_growth', 'Exports_growth', 'Imports_growth', 'CPI_growth',
       'Population_growth', 'Unemployment_rate', 'M.BCPI_growth',
       'M.BCNE_growth', 'M.ENER_growth', 'M.MTLS_growth', 'M.FOPR_growth',
       'M.AGRI_growth', 'M.FISH_growth', 'US_GDP_growth', 'TPUQ', 'TARIFFVOL',
       'tbill_3m'],
      dtype='object')

In [180]:
cols_to_drop = [
    'M.BCNE_growth','M.ENER_growth','M.MTLS_growth',
    'M.FOPR_growth','M.AGRI_growth','M.FISH_growth','Unnamed: 0'
]

growth_data = growth_data.drop(columns=cols_to_drop, errors='ignore')
print(growth_data.columns.tolist())
print(growth_data.shape)

['quarter', 'Consumption_growth', 'Government_growth', 'Investment_growth', 'Exports_growth', 'Imports_growth', 'CPI_growth', 'Population_growth', 'Unemployment_rate', 'M.BCPI_growth', 'US_GDP_growth', 'TPUQ', 'TARIFFVOL', 'tbill_3m']
(101, 14)


In [185]:
growth_data.tail()

,quarter,Consumption_growth,Government_growth,Investment_growth,Exports_growth,Imports_growth,CPI_growth,Population_growth,Unemployment_rate,M.BCPI_growth,US_GDP_growth,TPUQ,TARIFFVOL,tbill_3m
96,2024Q1,0.787989,1.444369,0.149942,-0.180090,-0.270371,0.419375,0.634590,5.900000,-0.727390,0.209861,71.450000,NaN,4.99
97,2024Q2,0.412387,1.364039,0.819987,-1.225688,0.002015,0.542911,0.652607,6.300000,7.162564,0.885486,91.500000,NaN,4.64
98,2024Q3,1.066146,1.341592,-0.452759,-0.153713,-0.248938,0.477674,0.663650,6.566667,-5.000157,0.824778,109.380000,NaN,4.02
99,2024Q4,1.190470,0.586024,2.172996,1.736514,0.617017,0.454733,0.561779,6.733333,-0.043604,0.459875,237.390000,NaN,3.15
100,2025Q1,0.141377,-0.107107,-1.261372,1.423985,0.885099,0.761317,0.193726,6.633333,4.293232,-0.162516,477.913598,NaN,2.64


In [182]:
endog_vars = [
    'Consumption_growth', 'Government_growth', 'Investment_growth',
    'Exports_growth', 'Imports_growth', 'CPI_growth',
    'Unemployment_rate',
    'Population_growth','tbill_3m'
]

exog_vars = [
    'TPUQ', 
    'TARIFFVOL', 'US_GDP_growth','M.BCPI_growth'
]


endog = growth_data[endog_vars]
exog = growth_data[exog_vars]

In [183]:
level_data.head()

,quarter,Consumption,Government,Investment,Exports,Imports,CPI,Population,Unemployment_rate,M.BCPI,M.BCNE,M.ENER,M.MTLS,M.FOPR,M.AGRI,M.FISH,GDPC1,TPUQ,TARIFFVOL
0,1992Q1,592278.0,313952,231808,265729,243333,83.300000,28181477,10.60000,246.386667,209.413333,429.476667,260.466667,240.893333,160.763333,599.166667,10236.435,42.225518,0.121545
1,1992Q2,594721.0,312184,222221,273246,245926,83.900000,28269699,11.00000,255.593333,214.206667,459.136667,263.570000,243.936667,168.550000,627.276667,10347.429,37.558292,0.094423
2,1992Q3,597899.0,312118,226708,277420,250009,84.200000,28371264,11.53333,257.000000,212.633333,474.433333,268.433333,244.690000,162.636667,534.963333,10449.673,50.093674,0.082377
3,1992Q4,600089.0,317134,217324,285010,244810,84.533333,28474177,11.73333,254.593333,209.013333,478.203333,246.290000,249.176667,162.053333,574.763333,10558.648,53.682379,0.057045
4,1993Q1,603297.0,314100,214916,295333,255921,85.100000,28533602,11.13333,266.456667,228.426667,455.486667,243.973333,301.050000,167.423333,672.156667,10576.275,50.748552,0.092414


In [189]:
level_data.columns

Index(['quarter', 'Consumption', 'Government', 'Investment', 'Exports',
       'Imports', 'CPI', 'Population', 'Unemployment_rate', 'M.BCPI', 'M.BCNE',
       'M.ENER', 'M.MTLS', 'M.FOPR', 'M.AGRI', 'M.FISH', 'GDPC1', 'TPUQ',
       'TARIFFVOL'],
      dtype='object')

In [187]:
# 0) Use your existing growth_data, endog_vars (unchanged)
exog_vars = ['TPUQ', 'US_GDP_growth', 'M.BCPI_growth']  # <- TARIFFVOL removed

# 1) Order and align
g = growth_data.copy()
g['quarter'] = pd.PeriodIndex(g['quarter'], freq='Q').astype(str)
g = g.sort_values('quarter')

Y_full = g[['quarter'] + endog_vars].dropna()
idx = pd.Index(Y_full['quarter'])

# Only exog are forward-filled if needed (safe for controls)
X_ff = (g[['quarter'] + exog_vars]
        .set_index('quarter')
        .replace([np.inf, -np.inf], np.nan)
        .ffill()
        .reindex(idx)
        .ffill())

df_use = pd.concat([Y_full.set_index('quarter'), X_ff], axis=1).dropna()
df_use = df_use.reset_index().rename(columns={'index': 'quarter'})

Y = df_use[endog_vars]
X = df_use[exog_vars]
last_q = df_use['quarter'].iloc[-1]  # should be 2025Q1 now

from statsmodels.tsa.api import VAR
import numpy as np
import pandas as pd

def fit_varx_const_and_forecast(Y, X, last_q, p, steps=8):
    res = VAR(Y, exog=X).fit(p, trend='c')           # VAR with constant
    exog_future = np.repeat(X.iloc[[-1]].to_numpy(), steps, axis=0)  # hold exog flat
    y_fcst = res.forecast(Y.values[-p:], steps=steps, exog_future=exog_future)
    fq = (pd.Period(last_q, 'Q') + np.arange(1, steps+1)).astype(str)
    out = pd.DataFrame(y_fcst, columns=[f"{c}_forecast" for c in Y.columns])
    out.insert(0, 'quarter', fq)
    return res, out

# 2) Run four separate lags p = 1,2,3,4
res_p1, fcst_p1 = fit_varx_const_and_forecast(Y, X, last_q, p=1, steps=8)
res_p2, fcst_p2 = fit_varx_const_and_forecast(Y, X, last_q, p=2, steps=8)
res_p3, fcst_p3 = fit_varx_const_and_forecast(Y, X, last_q, p=3, steps=8)
res_p4, fcst_p4 = fit_varx_const_and_forecast(Y, X, last_q, p=4, steps=8)

print("last training quarter:", last_q)     # expect '2025Q1'
print(fcst_p1['quarter'].head())            # should start at '2025Q2'

last training quarter: 2025Q1
0    2025Q2
1    2025Q3
2    2025Q4
3    2026Q1
4    2026Q2
Name: quarter, dtype: object


In [188]:
# assumes fcst_p1, fcst_p2, fcst_p3, fcst_p4 already exist
for p, df in zip([1, 2, 3, 4], [fcst_p1, fcst_p2, fcst_p3, fcst_p4]):
    df.to_csv(f"varx_const_p{p}_forecast.csv", index=False)

# Files created:
# - varx_const_p1_forecast.csv
# - varx_const_p2_forecast.csv
# - varx_const_p3_forecast.csv
# - varx_const_p4_forecast.csv

In [197]:
import pandas as pd
from pathlib import Path

# === settings ===
GROWTH_IS_PERCENT = True  # set False if your growth columns are already proportions (e.g., 0.007 = 0.7%)
in_dir  = Path(".")
out_dir = Path("./gdp_cpi_forecasts"); out_dir.mkdir(parents=True, exist_ok=True)

# load last observed levels (must have these columns)
lvl = pd.read_csv("level_data.csv")
lvl["quarter"] = pd.PeriodIndex(lvl["quarter"], freq="Q")
lvl = lvl.sort_values("quarter").reset_index(drop=True)
last = lvl.iloc[-1]
last_levels = {
    "Consumption": last["Consumption"],
    "Government":  last["Government"],
    "Investment":  last["Investment"],
    "Exports":     last["Exports"],
    "Imports":     last["Imports"],
    "CPI":         last["CPI"],
}

def to_factor(x):
    return 1 + (x/100.0 if GROWTH_IS_PERCENT else x)

def normalize_columns(df):
    """Rename *_forecast -> without suffix, and keep only needed growth cols."""
    rename = {}
    for base in ["Consumption_growth","Government_growth","Investment_growth",
                 "Exports_growth","Imports_growth","CPI_growth","Population_growth"]:
        col = f"{base}_forecast"
        if col in df.columns:
            rename[col] = base
    out = df.rename(columns=rename)
    need = ["quarter","Consumption_growth","Government_growth","Investment_growth",
            "Exports_growth","Imports_growth","CPI_growth"]
    missing = [c for c in need if c not in out.columns]
    if missing:
        raise ValueError(f"Missing in forecast file: {missing}")
    return out[need]

def levels_from_growth(fcst_df, last_levels):
    use = fcst_df.copy()
    use["quarter"] = pd.PeriodIndex(use["quarter"], freq="Q")
    use = use.sort_values("quarter").reset_index(drop=True)

    out = pd.DataFrame({"quarter": use["quarter"]})
    for comp in ["Consumption","Government","Investment","Exports","Imports"]:
        gcol = f"{comp}_growth"
        out[comp] = to_factor(use[gcol]).cumprod() * last_levels[comp]
    out["CPI"] = to_factor(use["CPI_growth"]).cumprod() * last_levels["CPI"]
    return out

for p in [1,2,3,4]:
    f = in_dir / f"varx_const_p{p}_forecast.csv"
    fcst_raw = pd.read_csv(f)
    fcst = normalize_columns(fcst_raw)
    lvl_path = levels_from_growth(fcst, last_levels)

    # aggregate GDP level
    lvl_path["gdp_level"] = (
        lvl_path["Consumption"] + lvl_path["Investment"] + lvl_path["Government"]
        + (lvl_path["Exports"] - lvl_path["Imports"])
    )
    lvl_path["cpi_level"] = lvl_path["CPI"]

    out = lvl_path[["quarter","gdp_level","cpi_level"]].copy()
    out["quarter"] = out["quarter"].astype(str)
    out.to_csv(out_dir / f"gdp_cpi_p{p}.csv", index=False)

print("Saved:", ", ".join([f"gdp_cpi_p{p}.csv" for p in [1,2,3,4]]))

Saved: gdp_cpi_p1.csv, gdp_cpi_p2.csv, gdp_cpi_p3.csv, gdp_cpi_p4.csv


In [199]:
import pandas as pd
import matplotlib.pyplot as plt

# Loop over lag orders
for p in range(1, 5):
    # Read forecast data
    df = pd.read_csv(f"gdp_cpi_forecast_p{p}.csv")

    # Convert quarter to datetime for cleaner plotting
    df["quarter"] = pd.PeriodIndex(df["quarter"], freq="Q").to_timestamp()

    # Plot CPI level forecast
    plt.figure(figsize=(10, 5))
    plt.plot(df["quarter"], df["CPI_level_forecast"], label="Forecast", color="orange", linestyle="--")
    plt.title(f"CPI Level Forecast (p={p})")
    plt.xlabel("Quarter")
    plt.ylabel("CPI Level")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"cpi_forecast_p{p}.png")
    plt.close()

    # Plot GDP level forecast
    plt.figure(figsize=(10, 5))
    plt.plot(df["quarter"], df["GDP_level_forecast"], label="Forecast", color="green", linestyle="--")
    plt.title(f"GDP Level Forecast (p={p})")
    plt.xlabel("Quarter")
    plt.ylabel("GDP Level (Million or Trillion CAD)")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"gdp_forecast_p{p}.png")
    plt.close()

In [202]:
import pandas as pd
import matplotlib.pyplot as plt

# Loop over lag orders
for p in range(1, 5):
    # Read forecast data
    df = pd.read_csv(f"gdp_cpi_forecast_p{p}.csv")

    # Convert quarter to datetime for cleaner plotting
    df["quarter"] = pd.PeriodIndex(df["quarter"], freq="Q").to_timestamp()

    # Plot CPI level forecast (blue line with points)
    plt.figure(figsize=(10, 5))
    plt.plot(
        df["quarter"],
        df["CPI_level_forecast"],
        label="Forecast",
        color="blue",
        linestyle="-",
        marker="o",
        markersize=4,
        linewidth=1.8,
    )
    plt.title(f"CPI Level Forecast (p={p})")
    plt.xlabel("Quarter")
    plt.ylabel("CPI Level")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"cpi_forecast_p{p}.png")
    plt.close()

    # Plot GDP level forecast (blue line with points)
    plt.figure(figsize=(10, 5))
    plt.plot(
        df["quarter"],
        df["GDP_level_forecast"],
        label="Forecast",
        color="blue",
        linestyle="-",
        marker="o",
        markersize=4,
        linewidth=1.8,
    )
    plt.title(f"GDP Level Forecast (p={p})")
    plt.xlabel("Quarter")
    plt.ylabel("GDP Level (Million or Trillion CAD)")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"gdp_forecast_p{p}.png")
    plt.close()